# 03 — Recover UART bytes from a noisy sampled waveform

**Goal:** reconstruct ASCII text from `data/noisy_uart.csv`. This dataset is **8N1, idle-high, 16 voltage samples per UART bit, LSB-first**. Voltage high is near 1 V; low is near 0 V; noise and a few spikes are added.

Run this notebook from `notebooks/` (the normal location when opening it from the zip). Only built-in modules are needed. You'll implement the decoder.

## Step 1: read the sample file

In C you'd `fopen` and loop over lines. Python's `with open(...) as f` closes the file for you. `csv.DictReader` returns rows with strings; use `float` to convert voltage values.

In [ ]:
import csv
with open('../data/noisy_uart.csv', newline='') as f:
    rows = csv.DictReader(f)
    samples = [float(row['voltage']) for row in rows]

samples_per_bit = 16
print('Number of samples:', len(samples))
print('First 8:', [round(x, 3) for x in samples[:8]])

## Step 2: decode a single known frame

The file begins with three idle-high bit periods, so the **first start bit is at index `3 * 16 = 48`**. Sample near each bit's center. The center of UART bit slot `k` is `start + (k + 0.5) * samples_per_bit`, where `k=0` is the start bit, slots 1–8 are data, and 9 is stop.

Average five samples around each center, then apply a 0.5-V threshold. Validate `frame[0] == 0` and `frame[9] == 1`. Reconstruct the byte LSB-first.

In [ ]:
def decode_frame(values, start, samples_per_bit=16):
    frame = []
    # TODO: loop over the 10 bit slots (start, 8 data, stop)
    # For each slot, average the five samples around the center.
    # Reject invalid start/stop bits by returning None.
    # Convert the 8 data bits to an integer, LSB first.
    raise NotImplementedError

assert decode_frame(samples, 48) == ord('H')
print('First decoded character:', chr(decode_frame(samples, 48)))

## Step 3: decode the whole message (known alignment)

In real hardware, frames aren't guaranteed to begin at sample 48. Start with the easy case: use the first known edge and advance **10 bit durations** to each next frame.

In [ ]:
def decode_aligned(values, first_start=48, samples_per_bit=16):
    decoded = []
    # TODO: collect successive frames until insufficient samples remain
    raise NotImplementedError

message = bytes(decode_aligned(samples))
assert message == b'Hi!'
print('Recovered bytes:', message)

## Bonus: find start edges automatically

A naive threshold crossing can trigger on data transitions or spikes. To find candidates, look for a **stable-high window followed by a stable-low window**, then validate the entire frame. After accepting a frame, skip approximately 10 bit durations.

Try scanning using the averages of `values[i-3:i]` and `values[i:i+4]`. There are many valid strategies. For real UART timing drift, you'd also consider resynchronizing each frame and estimating samples-per-bit.

In [ ]:
def decode_stream(values, samples_per_bit=16):
    decoded = []
    # TODO: detect start edge candidates; validate frames; advance by frame length
    raise NotImplementedError

assert bytes(decode_stream(samples)) == b'Hi!'
print('Stream decoder passed')

## Explain these in the interview

- Why five samples around the bit center instead of one?
- Why not low-pass across 20 samples when one bit lasts 16 samples?
- Why does `frame[k+1] << k` reconstruct an LSB-first UART byte?
- What happens with a parity error, missing stop bit, inverted logic, baud-rate mismatch, or sustained timing drift?
- What is the algorithm's runtime complexity? Approximately `O(number_of_samples)`.

**Next:** 04 — A carrier-modulated signal, which the interview email leaves open as a possibility.